# EDA — Telco Customer Churn

Análise exploratória do dataset IBM Telco Customer Churn.  
Objetivo: entender volume, qualidade, distribuição e data readiness antes da modelagem.

**Dataset:** 7.043 clientes, 21 variáveis, target binário `Churn` (Yes/No).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import DATA_BRONZE_DIR, TARGET_COL

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

df = pd.read_csv(DATA_BRONZE_DIR / 'telco_customer_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)

print(f'Shape: {df.shape}')
print(f'Churn rate: {df.Churn_bin.mean():.1%}')

## 1. Volume e Qualidade dos Dados

In [ ]:
print('=== TIPOS E NULOS ===')
info = pd.DataFrame({
    'dtype': df.dtypes,
    'nulls': df.isnull().sum(),
    'null_%': (df.isnull().sum() / len(df) * 100).round(2),
    'unique': df.nunique()
})
print(info[info['nulls'] > 0] if info['nulls'].sum() > 0 else 'Sem nulos após correção de TotalCharges')
print(f'\nDuplicatas: {df.duplicated().sum()}')

## 2. Distribuição do Target (Churn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['Churn'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição do Churn')
axes[0].set_ylabel('Clientes')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v}\n({v/len(df):.1%})', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Proporção')

plt.suptitle('Dataset desbalanceado: 26.5% de churners', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Features Numéricas — Distribuição por Churn

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(num_cols):
    for label, color in [('No', '#2ecc71'), ('Yes', '#e74c3c')]:
        axes[i].hist(df[df['Churn'] == label][col], alpha=0.6,
                     label=f'Churn={label}', color=color, bins=30)
    axes[i].set_title(col)
    axes[i].legend()

plt.suptitle('Distribuição das Features Numéricas por Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nEstatísticas por grupo:')
print(df.groupby('Churn')[num_cols].mean().round(1))

## 4. Features Categóricas — Taxa de Churn por Categoria

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'tenure_group']
df['tenure_group'] = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                             labels=['0-12m','13-24m','25-48m','49-72m'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn_bin'].mean().sort_values(ascending=False)
    bars = axes[i].bar(range(len(churn_rate)), churn_rate.values,
                       color=['#e74c3c' if v > 0.3 else '#f39c12' if v > 0.15 else '#2ecc71'
                              for v in churn_rate.values])
    axes[i].set_xticks(range(len(churn_rate)))
    axes[i].set_xticklabels(churn_rate.index, rotation=15, ha='right')
    axes[i].set_title(f'Taxa de Churn por {col}')
    axes[i].set_ylabel('Taxa de Churn')
    axes[i].axhline(df['Churn_bin'].mean(), color='navy', linestyle='--',
                    linewidth=1.5, label=f'Média: {df["Churn_bin"].mean():.1%}')
    axes[i].legend()
    for j, v in enumerate(churn_rate.values):
        axes[i].text(j, v + 0.01, f'{v:.1%}', ha='center', fontsize=9)

plt.suptitle('Taxa de Churn por Feature Categórica — Principais Drivers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Correlação entre Features Numéricas

In [ ]:
corr_df = df[num_cols + ['Churn_bin']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=ax, mask=mask, linewidths=0.5)
ax.set_title('Matriz de Correlação', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nInsight: TotalCharges e tenure são altamente correlacionados (r=0.83)')
print('TotalCharges ≈ tenure × MonthlyCharges — redundância intencional, capturada por charges_per_month')

## 6. Data Readiness — Conclusões para Modelagem

In [ ]:
print('=== DATA READINESS REPORT ===')
print(f'\n✅ Volume: {len(df):,} registros — suficiente para baseline, limitado para deep learning')
print(f'✅ Qualidade: 0 nulos após correção de TotalCharges (clientes com tenure=0)')
print(f'✅ Target: {df.Churn_bin.mean():.1%} positivos — desbalanceado, requer métricas adequadas (PR-AUC, F-beta)')
print(f'\n⚠️  Desafios identificados:')
print(f'   - TotalCharges como string com espaços: corrigido no pipeline silver')
print(f'   - Desbalanceamento 74/26: tratado com pos_weight=2.0 no BCEWithLogitsLoss')
print(f'   - TotalCharges ≈ tenure × MonthlyCharges: redundância tratada com feature engineering')
print(f'\n📊 Principais drivers de churn identificados:')
print(f'   1. Contrato Month-to-month: {df[df.Contract=="Month-to-month"].Churn_bin.mean():.1%} churn rate')
print(f'   2. Fiber optic: {df[df.InternetService=="Fiber optic"].Churn_bin.mean():.1%} churn rate')
print(f'   3. Clientes 0-12 meses: {df[df.tenure<=12].Churn_bin.mean():.1%} churn rate')
print(f'   4. Sem suporte técnico + sem segurança online: perfil de alto risco')